# Scraper Fiscal · Consulta Amigable (MEF) · Loreto, San Martín y Ucayali

**Un solo notebook** reemplaza a `DEV_Funcion_GobiernoNacional`, `DEV_Funcion_GobiernoRegional`,
`DEV_Funcion_Municipalidad`, `PIM_Funcion`, `PIM_Funcion_Municipalidad` y `DEV_Generica_Fuente`.
Guarda directo en Drive montado con **los mismos nombres y columnas** que esos scrapers,
así `Fiscal_Procesamiento_LOR_SMN_UCA.ipynb` y los CSV de CONTROL siguen funcionando igual.

| Bloque | Archivos que genera (en `Fiscal/Scrapeo/output/`) | Granularidad |
|---|---|---|
| `Funcion` | `DEVENGADO/Gobierno <Nivel>/DEV Funcion.<Nivel> AAAA - Mes MM - LOR - SMN - UCA.csv` | Mensual · genérica 2.6 por función (GL: además por municipalidad) |
| `Generica` | `.../DEV Generica.<Nivel> ...csv` | Mensual · todas las genéricas de gasto |
| `Fuente` | `.../DEV Fuente.<Nivel> ...csv` | Mensual · gasto total por fuente de financiamiento |
| `InvFuente` | `.../DEV InvFuente.<Nivel> ...csv` | Mensual · genérica 2.6 por fuente |
| `Deuda` | `.../DEV Deuda.<Nivel> ...csv` | Mensual · genérica 2.8: fila TOTAL + desglose en amortización e intereses (la clave de Sub-Genérica se autocalibra y se valida) |
| `PIM` | `PIM/PIM X FUNCIÓN/DATA/PIM Funcion AAAA - LOR - SMN - UCA.csv` | Anual · PIM 2.6 por función y nivel |
| `PIM_Municipalidad` | `PIM/PIM X FUNCIÓN X MUNICIPALIDAD/DATA/PIM Funcion.Municipalidad AAAA - LOR - SMN - UCA.csv` | Anual · PIM 2.6 por función y municipalidad |

Cada bloque deja su control en `DEVENGADO/CARPETA DE CONTROL/` o `PIM/.../CONTROL/`
(suma de meses vs vista anual del MEF; suma de niveles o municipios vs total).

**Rango de fechas:** `AÑO_INICIO`/`MES_INICIO` hasta `AÑO_FIN`/`MES_FIN` (`MES_FIN = None` = último mes
cerrado). Ej.: 2025/1 a 2026/None = Enero 2025 a Setiembre 2026. Los archivos, controles y PIM se generan
por año, igual que antes. Los meses ya raspados se sobrescriben con la cifra vigente del MEF, que puede
revisarse hacia atrás. Para un solo mes: `AÑO_INICIO = AÑO_FIN` y `MES_INICIO = MES_FIN = n`.

In [ ]:
# ═══ BLOQUE 1 · LIBRERÍAS Y DRIVE (una vez por sesión) ═══
!pip install requests beautifulsoup4 pandas numpy tqdm --quiet
import os, re, time, random, unicodedata, threading
import requests
import numpy as np
import pandas as pd
from datetime import date
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from IPython.display import display

from google.colab import drive
drive.mount("/content/drive")
RUTA_FISCAL = "/content/drive/MyDrive/Fiscal"
assert os.path.isdir(RUTA_FISCAL), f"No encuentro {RUTA_FISCAL}: revisa el nombre de la carpeta en Drive"
print("✅ Librerías listas · Drive montado")


In [ ]:
# ═══ BLOQUE 2 · CONFIGURACIÓN ═══
PRUEBA_RAPIDA = False          # True = corrida de validación: solo el mes MES_INICIO y solo Loreto (2-3 min). False = corrida completa.

AÑO_INICIO = 2026              # primer año a raspar
MES_INICIO = 1                 # primer mes de AÑO_INICIO
AÑO_FIN = 2026                 # último año a raspar
MES_FIN = None                 # último mes de AÑO_FIN · None = último mes cerrado disponible
# Ej.: 2025/1 a 2026/None = Enero 2025 a Setiembre 2026 (si hoy es octubre 2026). Los años intermedios van completos.
# Un solo mes: AÑO_INICIO = AÑO_FIN y MES_INICIO = MES_FIN.
INCLUIR_MES_EN_CURSO = False   # False = solo meses cerrados

NIVELES_A_CORRER = ["E", "R", "M"]      # E Nacional · R Regional · M Local
DEPARTAMENTOS_NACIONAL = "TODOS"        # detalle municipal (DEV Funcion.Municipalidad) y PIM (PIM Funcion y PIM Funcion.Municipalidad):
                                        # "TODOS" = los 25 departamentos · o una lista, p. ej. ["16", "22", "25"]. Lo demás, solo `departamentos`
BLOQUES_MENSUALES = ["Funcion", "Generica", "Fuente", "InvFuente", "Deuda"]
BLOQUES_ANUALES   = ["PIM", "PIM_Municipalidad"]    # quita "PIM_Municipalidad" si no lo necesitan ese mes

CONTROL_ANUAL = True           # suma de meses vs vista anual del MEF (bloques mensuales)
WORKERS = 4                    # consultas simultáneas (el MEF tolera bien 4-8)
PAUSA = (0.2, 0.6)             # segundos aleatorios entre consultas de un mismo hilo

# Claves numéricas de la URL de Consulta Amigable
CLAVE_GENERICA, CLAVE_MES, CLAVE_FUNCION, CLAVE_MUNICIPALIDAD = "24", "23", "8", "7"
# Claves ya identificadas en Consulta Amigable (no son Fuente): 1 nivel, 2 sector, 3 pliego, 4 ejecutora, 5 depto,
# 6 provincia, 7 municipalidad, 8 función, 13 meta, 21 depto meta, 23 mes, 24 genérica, 30 programa, 31 proyecto, 37 gob.loc./mancom.
CLAVES_CONOCIDAS = {"0", "1", "2", "3", "4", "5", "6", "7", "8", "13", "21", "23", "24", "30", "31", "37"}   # 37 = Gob.Loc./Mancom. (CONTEXTO_PROYECTO §4.2)
CANDIDATOS_FUENTE = ["14",   # verificada el 2026-10-09: 14 = Fuente de Financiamiento
                     "15", "16", "17", "18", "19", "20", "9", "10", "11", "12", "22", "25", "26", "27", "28", "29",
                     "32", "33", "34", "35", "36", "37", "38", "39", "40"]   # se autocalibra; pon primero la que ya conozcas
CANDIDATOS_SUBGENERICA = ["25", "26", "27", "28",   # Sub-Genérica y Sub-Genérica Detalle: se autocalibran y se VALIDAN por contenido
                          "9", "10", "11", "12", "15", "16", "17", "18", "19", "20", "22", "29",
                          "32", "33", "34", "35", "36", "37", "38", "39", "40"]   # si ya la conoces (F12 > Network), ponla primero
GENERICA_INVERSION = "6-2-6"   # 2.6 Adquisición de activos no financieros
GENERICA_DEUDA = "7-28"        # 2.8 Servicio de la deuda pública
PARAMS_EXTRA = {}

NIVELES_GOB = {
    "E": {"etiqueta": "E: GOBIERNO NACIONAL",    "corto": "GobNacional",   "carpeta": "Gobierno Nacional",  "claves_depto": ["21"],      "pim": "GOBIERNO NACIONAL"},
    "R": {"etiqueta": "R: GOBIERNOS REGIONALES", "corto": "GobRegional",   "carpeta": "Gobierno Regional",  "claves_depto": ["21"],      "pim": "GOBIERNOS REGIONALES"},
    "M": {"etiqueta": "M: GOBIERNOS LOCALES",    "corto": "Municipalidad", "carpeta": "Gobierno Municipal", "claves_depto": ["5", "21"], "pim": "GOBIERNOS LOCALES"},
}
departamentos = [
    {"codigo": "16", "departamento": "16: LORETO"},
    {"codigo": "22", "departamento": "22: SAN MARTÍN"},
    {"codigo": "25", "departamento": "25: UCAYALI"},
]
FUNCIONES = {f"{i:02d}": n for i, n in enumerate([
    "LEGISLATIVA", "RELACIONES EXTERIORES", "PLANEAMIENTO, GESTION Y RESERVA DE CONTINGENCIA",
    "DEFENSA Y SEGURIDAD NACIONAL", "ORDEN PUBLICO Y SEGURIDAD", "JUSTICIA", "TRABAJO", "COMERCIO", "TURISMO",
    "AGROPECUARIA", "PESCA", "ENERGIA", "MINERIA", "INDUSTRIA", "TRANSPORTE", "COMUNICACIONES", "AMBIENTE",
    "SANEAMIENTO", "VIVIENDA Y DESARROLLO URBANO", "SALUD", "CULTURA Y DEPORTE", "EDUCACION",
    "PROTECCION SOCIAL", "PREVISION SOCIAL", "DEUDA PUBLICA"], start=1)}
nombres_meses = ["Enero","Febrero","Marzo","Abril","Mayo","Junio","Julio","Agosto",
                 "Setiembre","Octubre","Noviembre","Diciembre"]
meses_map  = {str(i): f"{i}: {nombres_meses[i-1]}" for i in range(1, 13)}
TODOS_LOS_DEPARTAMENTOS = [{"codigo": c, "departamento": f"{c}: {n}"} for c, n in [
    ("01", "AMAZONAS"), ("02", "ÁNCASH"), ("03", "APURÍMAC"), ("04", "AREQUIPA"), ("05", "AYACUCHO"), ("06", "CAJAMARCA"),
    ("07", "CALLAO"), ("08", "CUSCO"), ("09", "HUANCAVELICA"), ("10", "HUÁNUCO"), ("11", "ICA"), ("12", "JUNÍN"),
    ("13", "LA LIBERTAD"), ("14", "LAMBAYEQUE"), ("15", "LIMA"), ("16", "LORETO"), ("17", "MADRE DE DIOS"), ("18", "MOQUEGUA"),
    ("19", "PASCO"), ("20", "PIURA"), ("21", "PUNO"), ("22", "SAN MARTÍN"), ("23", "TACNA"), ("24", "TUMBES"), ("25", "UCAYALI")]]
deptos_nacional = (TODOS_LOS_DEPARTAMENTOS if DEPARTAMENTOS_NACIONAL == "TODOS"
                   else [d for d in TODOS_LOS_DEPARTAMENTOS if d["codigo"] in DEPARTAMENTOS_NACIONAL])
deptos_map = {d["codigo"]: d["departamento"] for d in TODOS_LOS_DEPARTAMENTOS}
SUFIJO_REGIONES = "LOR - SMN - UCA"
def sufijo_regiones(lista):
    """Sufijo de los archivos según los departamentos que traen: LOR - SMN - UCA, TODOS LOS DEPARTAMENTOS o sus códigos."""
    cods = [d["codigo"] for d in lista]
    return SUFIJO_REGIONES if cods == ["16", "22", "25"] else "TODOS LOS DEPARTAMENTOS" if len(cods) == 25 else " - ".join(cods)
SUFIJO_PRUEBA = " - PRUEBA" if PRUEBA_RAPIDA else ""   # los archivos de prueba no pisan los reales

RUTA_DEV = os.path.join(RUTA_FISCAL, "Scrapeo", "output", "DEVENGADO")
RUTA_CTL = os.path.join(RUTA_DEV, "CARPETA DE CONTROL")
RUTA_PIM = os.path.join(RUTA_FISCAL, "Scrapeo", "output", "PIM", "PIM X FUNCIÓN")
RUTA_PIM_MUNI = os.path.join(RUTA_FISCAL, "Scrapeo", "output", "PIM", "PIM X FUNCIÓN X MUNICIPALIDAD")
for r in [RUTA_CTL, os.path.join(RUTA_PIM, "DATA"), os.path.join(RUTA_PIM, "CONTROL"),
          os.path.join(RUTA_PIM_MUNI, "DATA"), os.path.join(RUTA_PIM_MUNI, "CONTROL")]:
    os.makedirs(r, exist_ok=True)

hoy = date.today()
def ultimo_mes_disponible(anio):
    if anio < hoy.year:
        return 12
    tope = hoy.month if INCLUIR_MES_EN_CURSO else hoy.month - 1
    if tope < 1:
        raise ValueError(f"No hay meses cerrados aún en {anio}.")
    return tope

if not (AÑO_INICIO <= AÑO_FIN <= hoy.year):
    raise ValueError(f"Rango de años inválido ({AÑO_INICIO}-{AÑO_FIN}): AÑO_INICIO <= AÑO_FIN y sin años futuros.")
if MES_FIN is None:
    MES_FIN = ultimo_mes_disponible(AÑO_FIN)
if not (1 <= MES_INICIO <= 12 and 1 <= MES_FIN <= ultimo_mes_disponible(AÑO_FIN)):
    raise ValueError(f"Mes inválido. MES_FIN de {AÑO_FIN} puede ir de 1 a {ultimo_mes_disponible(AÑO_FIN)}.")
if AÑO_INICIO == AÑO_FIN and MES_INICIO > MES_FIN:
    raise ValueError(f"MES_INICIO ({MES_INICIO}) es mayor que MES_FIN ({MES_FIN}) en el mismo año.")
if PRUEBA_RAPIDA:
    AÑO_FIN, MES_FIN = AÑO_INICIO, MES_INICIO
    departamentos = departamentos[:1]
    deptos_nacional = departamentos[:1]
    print("🧪 PRUEBA RÁPIDA: solo", meses_map[str(MES_INICIO)], AÑO_INICIO, "y", departamentos[0]["departamento"],
          "(los archivos de este mes se sobrescriben solo con Loreto; vuelve a correr con PRUEBA_RAPIDA = False)")

def meses_de(anio):
    """Meses a raspar en un año del rango: el primer año arranca en MES_INICIO, el último termina en MES_FIN."""
    ini = MES_INICIO if anio == AÑO_INICIO else 1
    fin = MES_FIN if anio == AÑO_FIN else ultimo_mes_disponible(anio)
    return [str(m) for m in range(ini, fin + 1)]

PLAN = {a: meses_de(a) for a in range(AÑO_INICIO, AÑO_FIN + 1)}
def fijar_año(anio):
    """Deja AÑO, meses y cubre_anio del año que se procesa (los usan los nombres de archivo y los controles)."""
    global AÑO, meses, cubre_anio
    AÑO, meses = anio, PLAN[anio]
    cubre_anio = meses[0] == "1" and meses[-1] == "12"
fijar_año(AÑO_INICIO)

def etiqueta_meses(ms):
    nums = sorted(int(m) for m in ms)
    if len(nums) == 1:
        return f"Mes {nums[0]:02d}"
    if nums == list(range(nums[0], nums[-1] + 1)):
        return f"Meses {nums[0]:02d}-{nums[-1]:02d}"
    return "Meses " + ",".join(f"{m:02d}" for m in nums)

for a, ms in PLAN.items():
    print(f"Año {a} · meses {ms[0]}-{ms[-1]} ({len(ms)})")
print(f"Niveles {NIVELES_A_CORRER}")
print(f"Detalle municipal y PIM: {len(deptos_nacional)} departamento(s) · resto: {len(departamentos)}")
print(f"Bloques mensuales: {BLOQUES_MENSUALES} · anuales: {BLOQUES_ANUALES}")


In [ ]:
# ═══ BLOQUE 3 · NAVEGACIÓN EN CONSULTA AMIGABLE ═══
URL_BASE = "https://apps5.mineco.gob.pe/transparencia/mensual/Navegar_6.aspx"
_local = threading.local()
def sesion():
    if not hasattr(_local, "s"):
        s = requests.Session()
        s.headers.update({"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
                          "Referer": "https://apps5.mineco.gob.pe/transparencia/mensual/default.aspx"})
        _local.s = s
    return _local.s

FORMATOS = [
    {"nombre": "navegador GET", "metodo": "GET", "pre": {"_tgt": "frame", "_uhc": "yes"}, "post": {"ap": "ActProy"}},
    {"nombre": "GET + ap",      "metodo": "GET", "pre": {}, "post": {"ap": "ActProy"}},
    {"nombre": "POST + ap",     "metodo": "POST","pre": {}, "post": {"ap": "ActProy"}},
    {"nombre": "POST original", "metodo": "POST","pre": {}, "post": {}},
]
FORMATO = None
CLAVE_FUENTE = None
CLAVES_SUBGEN = []

def a_numero(txt):
    t = (txt or "").replace(",", "").replace("\xa0", "").strip()
    if t in ("", "-"):
        return 0.0
    try:
        return float(t)
    except ValueError:
        return np.nan

def normalizar(txt):
    t = unicodedata.normalize("NFKD", txt or "")
    return "".join(c for c in t if not unicodedata.combining(c)).lower().strip()

_FILTRO = re.compile(r"^(total|gen[eé]rica|nivel de gobierno|gob\.?\s?loc|departamento|provincia|distrito|"
                     r"funci[oó]n|mes\b|trimestre|sector|pliego|ejecutora|unidad ejecutora|fuente|rubro|categor|"
                     r"producto|divisi[oó]n|municipalidad|grupo funcional|sub-?gen[eé]rica|espec[ií]fica)", re.I)
_CODIGO = re.compile(r"^([\w\-\.]+)\s*:")

class Pagina:
    """filtros: breadcrumb [(etiqueta, devengado)] · filas: detalle [(valor_url, etiqueta, devengado, pim)]"""
    def __init__(self, html, url):
        self.url = url
        s = BeautifulSoup(html, "html.parser")
        self.tiene_tabla = s.find("table", class_="Data") is not None
        self.filtros, self.filas = [], []
        for tr in s.find_all("tr"):
            if tr.find("table"):
                continue
            celdas = tr.find_all("td")
            if len(celdas) < 8:
                continue
            dev = a_numero(celdas[-3].get_text(strip=True))   # … Devengado | Girado | Avance %
            if dev != dev:
                continue
            pim = a_numero(celdas[3].get_text(strip=True))    # radio | etiqueta | PIA | PIM | …
            textos = [" ".join(c.get_text(" ", strip=True).split()) for c in celdas[:3]]
            etiqueta = next((t for t in textos if t), "")
            radio = tr.find("input", attrs={"type": "radio"})
            if _FILTRO.match(etiqueta) and radio is None:
                self.filtros.append((etiqueta, dev, pim))
                continue
            m = _CODIGO.match(etiqueta)
            valor = (radio.get("value") if radio is not None else None) or (m.group(1) if m else None)
            if valor is None:
                continue
            valor = valor.split("/")[0].strip()        # el MEF pega los montos al valor del radio: "03///608658.94/…" (mensual) o "6-2-6/3083…/…" (anual)
            self.filas.append((valor, etiqueta, dev, pim))

def candidatos_de(valor, etiqueta):
    """Valores posibles para filtrar por una fila: valor del radio, código de la etiqueta y su último tramo."""
    m = _CODIGO.match(etiqueta)
    cods = ([m.group(1)] if m else []) + [valor] + ([m.group(1).split("-")[-1]] if m else [])
    return tuple(dict.fromkeys(c for c in cods if c))

def armar_params(anio, mes, nivel, depto, ruta, agrupar, formato):
    p = dict(formato["pre"]); p["0"] = ""
    if mes:
        p[CLAVE_MES] = mes
    if nivel:
        p["1"] = nivel
        for k in NIVELES_GOB[nivel]["claves_depto"]:
            p[k] = depto
    else:
        p["21"] = depto                        # sin nivel (control PIM): Departamento (Meta)
    for k, v in ruta:
        p[k] = v
    p[agrupar] = ""
    p["y"] = anio
    p.update(formato["post"]); p.update(PARAMS_EXTRA)
    return p

def consultar(anio, mes, nivel, depto, ruta, agrupar, formato=None, reintentos=4):
    formato = formato or FORMATO
    params = armar_params(anio, mes, nivel, depto, ruta, agrupar, formato)
    ultimo = ""
    for i in range(reintentos):
        try:
            r = sesion().request(formato["metodo"], URL_BASE, params=params, timeout=40)
            if r.status_code == 200:
                p = Pagina(r.content, r.url)
                if p.tiene_tabla:
                    time.sleep(random.uniform(*PAUSA))
                    return p, None
                ultimo = "SIN_TABLA"
            else:
                ultimo = f"HTTP {r.status_code}"
        except requests.RequestException as e:
            ultimo = f"{type(e).__name__}: {e}"
        time.sleep(min(30, 2 ** i + random.random()))
    return None, ultimo

def validar(p, nivel, depto):
    etiquetas = [normalizar(e) for e, _, _ in p.filtros]
    errores = []
    if nivel and not any(e.startswith("nivel de gobierno") and f" {nivel.lower()}:" in e for e in etiquetas):
        errores.append(f"la página no está filtrada por nivel {nivel}")
    if not any(e.startswith("departamento") and f"{depto}:" in e for e in etiquetas):
        errores.append(f"la página no corresponde al departamento {depto}")
    return errores

def filas_de(anio, mes, nivel, depto, ruta, agrupar, contexto, candidatos=None):
    """-> (filas, error, aviso): consulta validada por nivel/departamento y por el último filtro de la ruta.
    candidatos: valores alternativos para el último filtro de la ruta (se prueban hasta que el breadcrumb lo confirme)."""
    if candidatos and len(candidatos) > 1 and ruta:
        ultimo_err = None
        for cand in candidatos:
            filas, err, aviso = filas_de(anio, mes, nivel, depto, list(ruta[:-1]) + [(ruta[-1][0], cand)], agrupar, contexto)
            if not (err and ("no aplicó el filtro" in err or "SIN_TABLA" in err)):
                return filas, err, aviso
            ultimo_err = err
        return [], ultimo_err, None
    p, err = consultar(anio, mes, nivel, depto, ruta, agrupar)
    if err:
        return [], f"{contexto}: {err}", None
    e = validar(p, nivel, depto)
    if e:
        return [], f"{contexto}: {'; '.join(e)}", None
    if ruta and p.filtros:
        esperado = str(ruta[-1][1])
        if not any(esperado in f or esperado.split("-")[-1] in f.split(":")[0] for f, _, _ in p.filtros):
            return [], f"{contexto}: la URL no aplicó el filtro {ruta[-1]}", None
    if not p.filas:
        if p.filtros and p.filtros[-1][1] == 0 and p.filtros[-1][2] == 0:
            return [], None, None                 # filtro con todo en 0: no hay detalle
        return [], f"{contexto}: sin filas de detalle", None
    aviso = None
    if p.filtros:
        total, suma = p.filtros[-1][1], sum(f[2] for f in p.filas)
        if abs(suma - total) > max(1, len(p.filas)):
            aviso = f"CUADRE {contexto}: suma {suma:,.0f} vs {p.filtros[-1][0][:40]} {total:,.0f}"
    return p.filas, None, aviso

_RE_DEUDA = re.compile(r"servicio de la deuda", re.I)
_RE_INVERSION = re.compile(r"activos no financieros", re.I)

# ── Calibración (formato de URL, clave de Fuente, cadena de sub-genéricas) ───
def calibrar(anio, mes, depto):
    global FORMATO, CLAVE_FUENTE, CLAVES_SUBGEN
    print("🔗 Calibrando formato de URL…")
    for f in FORMATOS:
        p, err = consultar(anio, mes, "E", depto, [], CLAVE_GENERICA, formato=f, reintentos=2)
        n = len(p.filas) if p else 0
        print(f"   · {f['nombre']:<15} -> {err or f'{n} filas de detalle'}")
        if p and n and not validar(p, "E", depto):
            FORMATO = f; break
    if FORMATO is None:
        raise RuntimeError("Consulta Amigable no devolvió la tabla de detalle con ningún formato de URL.")
    # códigos de URL de las genéricas 2.6 y 2.8 tal como los usa el MEF (p. ej. "6-2-6", "7-2-8")
    global GENERICA_INVERSION, GENERICA_DEUDA
    encontrado_deuda = False
    for nivel_cal in ("E", "R", "M"):           # vista anual: ahí aparecen todas las genéricas con gasto en el año
        pg, err = consultar(anio, "", nivel_cal, depto, [], CLAVE_GENERICA, reintentos=2)
        for v, et, _, _ in (pg.filas if pg else []):
            if _RE_INVERSION.search(normalizar(et)) and v != GENERICA_INVERSION:
                print(f"   genérica 2.6: el MEF usa '{v}' (antes '{GENERICA_INVERSION}')"); GENERICA_INVERSION = v
            if _RE_DEUDA.search(normalizar(et)):
                GENERICA_DEUDA, encontrado_deuda = v, True
        if encontrado_deuda:
            break
    print(f"   genérica inversión = {GENERICA_INVERSION} · genérica deuda = {GENERICA_DEUDA}"
          + ("" if encontrado_deuda else " (no apareció en el listado; sin verificar)"))

    if any(b in BLOQUES_MENSUALES for b in ("Fuente", "InvFuente")):
        print("🔗 Calibrando clave de Fuente de Financiamiento…")
        rx = re.compile(r"recursos ordinarios|directamente recaudados|credito|donaciones|recursos determinados", re.I)
        for k in CANDIDATOS_FUENTE:
            if k in CLAVES_CONOCIDAS:
                continue
            p, err = consultar(anio, mes, "E", depto, [], k, reintentos=1)
            if err or not p.filas:
                print(f"   · clave {k:<3} -> {err or 'sin filas'}"); continue
            aciertos = sum(bool(rx.search(normalizar(et))) for _, et, _, _ in p.filas)
            print(f"   · clave {k:<3} -> {len(p.filas)} filas, {aciertos} parecen fuentes: {' | '.join(et[:28] for _, et, _, _ in p.filas[:3])}")
            if aciertos >= max(2, len(p.filas) // 2):
                CLAVE_FUENTE = k; break
        if CLAVE_FUENTE is None:
            raise RuntimeError("Ninguna clave candidata listó fuentes de financiamiento. Copia la salida de arriba "
                               "(qué listó cada clave) o mira en F12 > Network el parámetro que usa la web al elegir "
                               "'Fuente de Financiamiento' y ponlo primero en CANDIDATOS_FUENTE.")
        print(f"   clave Fuente = {CLAVE_FUENTE}")

    if "Deuda" in BLOQUES_MENSUALES:
        calibrar_deuda(anio)

_RE_SUBDEUDA = re.compile(r"deuda|amortiz|interes|comision")
_RE_HOJA_DEUDA = re.compile(r"amortiz|interes|comision")

def calibrar_deuda(anio):
    """Busca la cadena de claves Genérica 2.8 -> Sub-Genérica -> Sub-Genérica Detalle.
    CORRECCIÓN: antes se aceptaba cualquier clave que devolviera filas (aunque listara otra dimensión) y se bajaba
    solo por la primera fila; con una cadena equivocada todas las filas salían como OTRO o con error y la deuda
    quedaba vacía. Ahora una clave solo se acepta si TODAS sus filas son rubros de deuda, se prueba con los tres
    departamentos, niveles R/M/E, el año y el anterior, y si no se encuentra la cadena el bloque Deuda avisa con
    error en vez de guardar datos sin verificar."""
    global CLAVES_SUBGEN
    print(f"🔗 Calibrando sub-niveles de la genérica {GENERICA_DEUDA}…")
    candidatos = [k for k in dict.fromkeys(CANDIDATOS_SUBGENERICA) if k not in CLAVES_CONOCIDAS and k != CLAVE_FUENTE]
    # Pocas consultas (Incapsula, CONTEXTO_PROYECTO §4.3): una clave que da error o lista otra dimensión se descarta
    # para siempre en esa profundidad, y solo se calibra donde la genérica 2.8 tiene monto.
    descartadas = {0: set(), 1: set(), 2: set()}
    for anio_cal in (anio, anio - 1):
        for depto in [d["codigo"] for d in departamentos]:
            for nivel_cal in ("R", "M", "E"):
                pg, err = consultar(anio_cal, "", nivel_cal, depto, [], CLAVE_GENERICA, reintentos=2)
                if err or not any(_RE_DEUDA.search(normalizar(et)) and (dev or pim) for _, et, dev, pim in pg.filas):
                    continue                              # sin deuda en ese departamento/nivel/año
                ruta, cadena = [(CLAVE_GENERICA, GENERICA_DEUDA)], []
                for prof in range(3):
                    hallado = None
                    for k in candidatos:
                        if k in cadena or k in descartadas[prof]:
                            continue
                        p, err = consultar(anio_cal, "", nivel_cal, depto, ruta, k, reintentos=1)
                        if err or (p.filas and validar(p, nivel_cal, depto)):
                            descartadas[prof].add(k); continue
                        if not p.filas:
                            continue
                        etiquetas = [normalizar(re.sub(r"^[\w\-\.]+\s*:\s*", "", et)) for _, et, _, _ in p.filas]
                        if all(_RE_SUBDEUDA.search(e) for e in etiquetas):
                            hallado = (k, p); break
                        descartadas[prof].add(k)          # lista otra dimensión
                    if hallado is None:
                        break
                    k, p = hallado
                    cadena.append(k)
                    print(f"   · {anio_cal} {deptos_map[depto]} nivel {nivel_cal} clave {k:<3} -> "
                          + " | ".join(et[:40] for _, et, _, _ in p.filas[:4]))
                    if any(_RE_HOJA_DEUDA.search(normalizar(et)) for _, et, _, _ in p.filas):
                        CLAVES_SUBGEN = cadena
                        print(f"   ✅ cadena Sub-genérica = {CLAVES_SUBGEN}")
                        return True
                    fila = max(p.filas, key=lambda f: abs(f[2]) + abs(f[3]))   # baja por la fila con más monto
                    ruta = ruta + [(k, fila[0])]
                if cadena:
                    print(f"   · {anio_cal} {deptos_map[depto]} nivel {nivel_cal}: la cadena {cadena} no llegó a intereses/amortización")
    CLAVES_SUBGEN = []
    print("   ⚠️ No se encontró la clave de Sub-Genérica. El bloque Deuda guardará solo el TOTAL de la genérica 2.8\n"
          "      (el procesamiento igual llena SERVICIO DE LA DEUDA). Para el desglose: en Consulta Amigable entra a\n"
          "      Genérica 2-8 > Sub-Genérica, copia de la URL (F12 > Network) la clave numérica y ponla primera en CANDIDATOS_SUBGENERICA.")
    return False


In [ ]:
# ═══ BLOQUE 4 · EXTRACTORES ═══
def base(anio, mes, nivel, depto):
    return {"AÑO": anio, "MES": meses_map.get(mes, "ANUAL"), "NIVEL_GOBIERNO": NIVELES_GOB[nivel]["etiqueta"],
            "DEPARTAMENTO": deptos_map[depto], "COD_DEPTO": depto}

def etiqueta_funcion(et):
    m = _CODIGO.match(et)
    cod = m.group(1) if m else et[:2]
    return cod, (f"{cod}: {FUNCIONES[cod]}" if cod in FUNCIONES else et)

def extraer_funcion(anio, mes, nivel, depto):
    """Genérica 2.6 por función. En GL baja además a municipalidad (igual que DEV_Funcion_Municipalidad)."""
    ctx = f"Función {deptos_map[depto]}"
    filas, err, aviso = filas_de(anio, mes, nivel, depto, [(CLAVE_GENERICA, GENERICA_INVERSION)], CLAVE_FUNCION, ctx)
    if err:
        return [], err, None
    regs, errores, avisos = [], [], [aviso] if aviso else []
    entidad = "GOBIERNO NACIONAL" if nivel == "E" else f"GOBIERNO REGIONAL DE {deptos_map[depto].split(': ')[1]}"
    for valor, et, dev, _ in filas:
        cod, nombre = etiqueta_funcion(et)
        if nivel != "M":
            regs.append(dict(base(anio, mes, nivel, depto), ENTIDAD=entidad, COD_FUNCION=cod, FUNCION=nombre, DEVENGADO=dev))
            continue
        sub, e2, a2 = filas_de(anio, mes, nivel, depto, [(CLAVE_GENERICA, GENERICA_INVERSION), (CLAVE_FUNCION, cod)],
                               CLAVE_MUNICIPALIDAD, f"{ctx} > {nombre[:30]}", candidatos=candidatos_de(valor, et))
        if e2:
            errores.append(e2); continue
        if a2:
            avisos.append(a2)
        for _, mun, dmun, _ in sub:
            regs.append(dict(base(anio, mes, nivel, depto), COD_FUNCION=cod, FUNCION=nombre, MUNICIPALIDAD=mun, DEVENGADO=dmun))
    return regs, ("; ".join(errores) or None), ("; ".join(avisos) or None)

def extraer_generica(anio, mes, nivel, depto):
    filas, err, aviso = filas_de(anio, mes, nivel, depto, [], CLAVE_GENERICA, f"Genérica {deptos_map[depto]}")
    return [dict(base(anio, mes, nivel, depto), COD_GENERICA=v, GENERICA=et, DEVENGADO=dev) for v, et, dev, _ in filas], err, aviso

def extraer_fuente(anio, mes, nivel, depto, generica=None):
    ruta = [(CLAVE_GENERICA, generica)] if generica else []
    filas, err, aviso = filas_de(anio, mes, nivel, depto, ruta, CLAVE_FUENTE, f"Fuente{' 2.6' if generica else ''} {deptos_map[depto]}")
    return [dict(base(anio, mes, nivel, depto), GENERICA=generica or "TODAS", COD_FUENTE=v, FUENTE=et, DEVENGADO=dev)
            for v, et, dev, _ in filas], err, aviso

def clasificar_deuda(etiquetas):
    t = normalizar(" > ".join(etiquetas))
    return "AMORTIZACION" if "amortiz" in t else "INTERESES Y COMISIONES" if ("interes" in t or "comision" in t) else "OTRO"

def extraer_deuda(anio, mes, nivel, depto):
    ctx = f"Deuda {deptos_map[depto]}"
    filas, err, _ = filas_de(anio, mes, nivel, depto, [], CLAVE_GENERICA, ctx)
    if err:
        return [], err, None
    total = next((dev for v, et, dev, _ in filas if v == GENERICA_DEUDA or _RE_DEUDA.search(normalizar(et))), 0.0)
    if total == 0:
        return [], None, None
    # CORRECCIÓN: la fila TOTAL se guarda siempre, aunque falle el desglose (el procesamiento la usa como subtotal)
    total_reg = dict(base(anio, mes, nivel, depto), GENERICA=GENERICA_DEUDA, RUTA="TOTAL GENÉRICA 2.8", CLASIFICACION="TOTAL", DEVENGADO=total)
    if not CLAVES_SUBGEN:
        return [total_reg], f"{ctx}: sin clave de Sub-Genérica calibrada (solo se guardó el total)", None
    regs, errores, avisos = [], [], []
    def bajar(ruta, etiquetas, i):
        clave = CLAVES_SUBGEN[i]
        sub, e, a = filas_de(anio, mes, nivel, depto, ruta, clave, f"{ctx} {' > '.join(etiquetas)[:50]}",
                             candidatos=candidatos_de(ruta[-1][1], etiquetas[-1]) if etiquetas else None)
        if e:
            errores.append(e); return
        if a:
            avisos.append(a)
        for v, et, dev, _ in sub:
            cadena = etiquetas + [et]; clase = clasificar_deuda(cadena)
            if clase != "OTRO" or i + 1 >= len(CLAVES_SUBGEN):
                regs.append(dict(base(anio, mes, nivel, depto), GENERICA=GENERICA_DEUDA, RUTA=" > ".join(cadena), CLASIFICACION=clase, DEVENGADO=dev))
            else:
                bajar(ruta + [(clave, v)], cadena, i + 1)   # v ya viene limpio; filas_de prueba alternativas si no aplica
    bajar([(CLAVE_GENERICA, GENERICA_DEUDA)], [], 0)
    suma = sum(r["DEVENGADO"] for r in regs)
    if abs(suma - total) > max(1, len(regs)):
        avisos.append(f"CUADRE deuda {deptos_map[depto]}: hojas {suma:,.0f} vs genérica {total:,.0f}")
    otros = [r["RUTA"] for r in regs if r["CLASIFICACION"] == "OTRO"]
    if otros:
        avisos.append(f"Deuda {deptos_map[depto]}: {len(otros)} fila(s) sin clasificar como intereses/amortización: {' | '.join(otros)[:120]}")
    return [total_reg] + regs, ("; ".join(errores) or None), ("; ".join(avisos) or None)

EXTRACTORES = {
    "Funcion":   extraer_funcion,
    "Generica":  extraer_generica,
    "Fuente":    lambda a, m, n, d: extraer_fuente(a, m, n, d),
    "InvFuente": lambda a, m, n, d: extraer_fuente(a, m, n, d, GENERICA_INVERSION),
    "Deuda":     extraer_deuda,
}
def columnas(bloque, nivel):
    if bloque == "Funcion":
        return (["AÑO", "MES", "DEPARTAMENTO", "FUNCION", "MUNICIPALIDAD", "DEVENGADO"] if nivel == "M"
                else ["AÑO", "MES", "NIVEL_GOBIERNO", "DEPARTAMENTO", "ENTIDAD", "FUNCION", "DEVENGADO"])
    return {"Generica":  ["AÑO", "MES", "NIVEL_GOBIERNO", "DEPARTAMENTO", "COD_GENERICA", "GENERICA", "DEVENGADO"],
            "Fuente":    ["AÑO", "MES", "NIVEL_GOBIERNO", "DEPARTAMENTO", "GENERICA", "COD_FUENTE", "FUENTE", "DEVENGADO"],
            "InvFuente": ["AÑO", "MES", "NIVEL_GOBIERNO", "DEPARTAMENTO", "GENERICA", "COD_FUENTE", "FUENTE", "DEVENGADO"],
            "Deuda":     ["AÑO", "MES", "NIVEL_GOBIERNO", "DEPARTAMENTO", "GENERICA", "RUTA", "CLASIFICACION", "DEVENGADO"]}[bloque]
CLAVE_CONTROL = {"Funcion": "COD_FUNCION", "Generica": "COD_GENERICA", "Fuente": "COD_FUENTE", "InvFuente": "COD_FUENTE", "Deuda": "CLASIFICACION"}

def deptos_de(bloque, nivel):
    """Departamentos que se raspan en cada bloque: el detalle municipal (Funcion en GL) va con deptos_nacional."""
    return deptos_nacional if (bloque == "Funcion" and nivel == "M") else departamentos

def nombre_salida(bloque, nivel, ms):
    return f"DEV {bloque}.{NIVELES_GOB[nivel]['corto']} {AÑO} - {etiqueta_meses(ms)} - {sufijo_regiones(deptos_de(bloque, nivel))}{SUFIJO_PRUEBA}.csv"

def nombre_control(bloque, nivel):
    corto = NIVELES_GOB[nivel]["corto"]
    sufijo = "" if cubre_anio else f" - {etiqueta_meses(meses)}"
    if bloque == "Funcion":   # mismos nombres que los scrapers anteriores
        return f"CONTROL {corto} {AÑO}{sufijo}{' - ' + sufijo_regiones(deptos_nacional) if nivel == 'M' else ''}{SUFIJO_PRUEBA}.csv"
    return f"CONTROL {bloque} {corto} {AÑO}{sufijo}{SUFIJO_PRUEBA}.csv"

# ── PIM (anual) ──────────────────────────────────────────────────────────────
def extraer_pim(anio, nivel, depto):
    """PIM de la genérica 2.6 por función (nivel=None -> total de todos los niveles, para el control)."""
    filas, err, _ = filas_de(anio, "", nivel, depto, [(CLAVE_GENERICA, GENERICA_INVERSION)], CLAVE_FUNCION,
                             f"PIM {nivel or 'TODOS'} {deptos_map[depto]}")
    regs = []
    for v, et, _, pim in filas:
        cod, nombre = etiqueta_funcion(et)
        regs.append({"AÑO": anio, "DEPARTAMENTO": deptos_map[depto], "NIVEL DE GOBIERNO": NIVELES_GOB[nivel]["pim"] if nivel else "TODOS",
                     "COD_NIVEL": nivel or "", "COD_FUNCION": cod, "FUNCION": nombre, "PIM": pim, "_valor": v})
    return regs, err

def extraer_pim_municipal(anio, depto):
    regs, errores = [], []
    filas, err, _ = filas_de(anio, "", "M", depto, [(CLAVE_GENERICA, GENERICA_INVERSION)], CLAVE_FUNCION, f"PIM muni {deptos_map[depto]}")
    if err:
        return [], err
    for v, et, _, _ in filas:
        cod, nombre = etiqueta_funcion(et)
        sub, e2, _ = filas_de(anio, "", "M", depto, [(CLAVE_GENERICA, GENERICA_INVERSION), (CLAVE_FUNCION, cod)],
                              CLAVE_MUNICIPALIDAD, f"PIM muni {deptos_map[depto]} > {nombre[:30]}", candidatos=candidatos_de(v, et))
        if e2:
            errores.append(e2); continue
        for _, mun, _, pim in sub:
            regs.append({"AÑO": anio, "DEPARTAMENTO": deptos_map[depto], "COD_FUNCION": cod, "FUNCION": nombre, "MUNICIPALIDAD": mun, "PIM": pim})
    return regs, ("; ".join(errores) or None)


In [ ]:
# ═══ BLOQUE 5 · EJECUCIÓN (un ciclo por año del rango) ═══
fijar_año(AÑO_FIN)                       # calibra con el año más reciente: claves y códigos vigentes del MEF
calibrar(AÑO, meses[-1], departamentos[0]["codigo"])
for _anio in PLAN:
    fijar_año(_anio)
    print("\n" + "█" * 70 + f"\n{AÑO}: meses {meses[0]}-{meses[-1]}")
    errores_totales, avisos_totales, archivos, resumen = [], [], [], []

    def tarea(bloque, nivel, mes, d):
        regs, err, aviso = EXTRACTORES[bloque](AÑO, mes, nivel, d["codigo"])
        return bloque, nivel, mes, d["departamento"], regs, err, aviso

    # ── 5a. Bloques mensuales: por mes, todas las combinaciones nivel x bloque x departamento en paralelo ──
    registros = {(b, n): [] for b in BLOQUES_MENSUALES for n in NIVELES_A_CORRER}
    for mes in meses:
        etiqueta = meses_map[mes]
        tareas = [(b, n, mes, d) for n in NIVELES_A_CORRER for b in BLOQUES_MENSUALES for d in deptos_de(b, n)]
        por_clave, errs_mes = {k: [] for k in registros}, 0
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futuros = [ex.submit(tarea, *t) for t in tareas]
            for f in tqdm(as_completed(futuros), total=len(futuros), desc=f"{AÑO} {etiqueta}", leave=False):
                b, n, m, dep, regs, err, aviso = f.result()
                por_clave[(b, n)].extend(regs)
                if err:
                    errs_mes += 1; errores_totales.append({"AÑO": AÑO, "MES": etiqueta, "NIVEL": n, "BLOQUE": b, "DEPARTAMENTO": dep, "ERROR": err})
                if aviso:
                    avisos_totales.append({"AÑO": AÑO, "MES": etiqueta, "NIVEL": n, "BLOQUE": b, "DEPARTAMENTO": dep, "AVISO": aviso})
        fila = {"MES": etiqueta}
        for (b, n), regs in por_clave.items():
            registros[(b, n)].extend(regs)
            fila[f"{b}.{n}"] = len(regs)
            if regs:
                ruta = os.path.join(RUTA_DEV, NIVELES_GOB[n]["carpeta"]); os.makedirs(ruta, exist_ok=True)
                archivo = os.path.join(ruta, nombre_salida(b, n, [mes]))
                pd.DataFrame(regs)[columnas(b, n)].to_csv(archivo, index=False, encoding="utf-8-sig")
                archivos.append(archivo)
        fila["ERRORES"] = errs_mes; fila["ESTADO"] = "OK" if not errs_mes else "INCOMPLETO"
        resumen.append(fila)
        print(f"✅ {etiqueta}: " + " · ".join(f"{k}={v}" for k, v in fila.items() if k not in ("MES", "ESTADO")))

    # ── 5b. Control anual de los bloques mensuales ──
    if CONTROL_ANUAL and meses:
        print("\n🔍 Control anual contra la vista anual de Consulta Amigable…")
        for (b, n), regs in registros.items():
            df_total = pd.DataFrame(regs)
            if df_total.empty:
                continue
            clave, filas = CLAVE_CONTROL[b], []
            with ThreadPoolExecutor(max_workers=WORKERS) as ex:   # la vista anual de cada departamento, en paralelo
                refs = list(ex.map(lambda d: EXTRACTORES[b](AÑO, "", n, d["codigo"]), deptos_de(b, n)))
            for d, (ref_regs, err, _) in zip(deptos_de(b, n), refs):
                if err or not ref_regs:
                    print(f"   ❌ Referencia anual {b}.{n} {d['departamento']}: {err or 'sin filas'}"); continue
                ref = pd.DataFrame(ref_regs).groupby(clave)["DEVENGADO"].sum()
                sub = df_total[df_total["COD_DEPTO"] == d["codigo"]]
                suma = sub.groupby(clave)["DEVENGADO"].sum()
                ult = sub[sub["MES"] == meses_map[meses[-1]]].groupby(clave)["DEVENGADO"].sum()
                for cod in sorted(set(ref.index) | set(suma.index)):
                    filas.append({"AÑO": AÑO, "DEPARTAMENTO": d["departamento"], clave: cod, "DEV_REF_ANUAL": float(ref.get(cod, 0.0)),
                                  "DEV_SUMA_MESES": float(suma.get(cod, 0.0)), "DEV_ULTIMO_MES": float(ult.get(cod, 0.0))})
            ctl = pd.DataFrame(filas)
            if ctl.empty:
                continue
            tol = max(5, len(meses) * 5)
            ctl["DIF_SUMA"] = ctl["DEV_SUMA_MESES"] - ctl["DEV_REF_ANUAL"]
            ctl["DIF_ULTIMO"] = ctl["DEV_ULTIMO_MES"] - ctl["DEV_REF_ANUAL"]
            ok_suma, ok_ult = ctl["DIF_SUMA"].abs().le(tol).all(), ctl["DIF_ULTIMO"].abs().le(tol).all()
            col = "DIF_SUMA" if (ok_suma or not ok_ult) else "DIF_ULTIMO"
            semantica = "MENSUAL" if ok_suma else "ACUMULADO (NO sumar meses)" if ok_ult else ("AÑO INCOMPLETO" if not cubre_anio else "NO CUADRA")
            ctl["ESTADO"] = np.where(ctl[col].abs().le(tol), "OK", "REVISAR" if cubre_anio else "PARCIAL")
            ctl["TOLERANCIA_S/"] = tol; ctl["FECHA_EXTRACCION"] = date.today().isoformat()
            archivo = os.path.join(RUTA_CTL, nombre_control(b, n))
            ctl.to_csv(archivo, index=False, encoding="utf-8-sig"); archivos.append(archivo)
            print(f"   {b + '.' + n:<24} {semantica:<28} {(ctl['ESTADO'] == 'OK').sum()}/{len(ctl)} OK -> {os.path.basename(archivo)}")

    # ── 5c. PIM anual por función (3 niveles + control contra el total) ──
    if "PIM" in BLOQUES_ANUALES:
        print("\n📘 PIM por función…")
        regs, ctl_filas = [], []
        for d in deptos_nacional:
            por_nivel = {}
            for n in ("E", "R", "M"):
                r, err = extraer_pim(AÑO, n, d["codigo"])
                if err:
                    errores_totales.append({"AÑO": AÑO, "MES": "ANUAL", "NIVEL": n, "BLOQUE": "PIM", "DEPARTAMENTO": d["departamento"], "ERROR": err})
                por_nivel[n] = {x["COD_FUNCION"]: x["PIM"] for x in r}; regs += r
            ref, err = extraer_pim(AÑO, None, d["codigo"])
            if err:
                errores_totales.append({"AÑO": AÑO, "MES": "ANUAL", "NIVEL": "TODOS", "BLOQUE": "PIM", "DEPARTAMENTO": d["departamento"], "ERROR": err})
            ref = {x["COD_FUNCION"]: x["PIM"] for x in ref}
            for cod in sorted(set(ref) | set().union(*[set(v) for v in por_nivel.values()])):
                fila = {"AÑO": AÑO, "DEPARTAMENTO": d["departamento"], "COD_FUNCION": cod, "FUNCION": f"{cod}: {FUNCIONES.get(cod, '')}",
                        "PIM_GOBIERNO_NACIONAL": por_nivel["E"].get(cod, 0.0), "PIM_GOBIERNOS_LOCALES": por_nivel["M"].get(cod, 0.0),
                        "PIM_GOBIERNOS_REGIONALES": por_nivel["R"].get(cod, 0.0)}
                fila["PIM_SUMA_NIVELES"] = fila["PIM_GOBIERNO_NACIONAL"] + fila["PIM_GOBIERNOS_LOCALES"] + fila["PIM_GOBIERNOS_REGIONALES"]
                fila["PIM_REF_TOTAL"] = ref.get(cod, 0.0); fila["DIF"] = fila["PIM_SUMA_NIVELES"] - fila["PIM_REF_TOTAL"]
                fila["ESTADO"] = "OK" if abs(fila["DIF"]) <= 3 else "REVISAR"; fila["TOLERANCIA_S/"] = 3; fila["FECHA_EXTRACCION"] = date.today().isoformat()
                ctl_filas.append(fila)
        if regs:
            df = pd.DataFrame(regs).sort_values(["DEPARTAMENTO", "COD_NIVEL", "COD_FUNCION"])
            archivo = os.path.join(RUTA_PIM, "DATA", f"PIM Funcion {AÑO} - {sufijo_regiones(deptos_nacional)}{SUFIJO_PRUEBA}.csv")
            df[["AÑO", "DEPARTAMENTO", "NIVEL DE GOBIERNO", "FUNCION", "PIM"]].to_csv(archivo, index=False, encoding="utf-8-sig"); archivos.append(archivo)
            ctl = pd.DataFrame(ctl_filas)
            archivo_ctl = os.path.join(RUTA_PIM, "CONTROL", f"CONTROL PIM Funcion {AÑO} - {sufijo_regiones(deptos_nacional)}{SUFIJO_PRUEBA}.csv")
            ctl.to_csv(archivo_ctl, index=False, encoding="utf-8-sig"); archivos.append(archivo_ctl)
            print(f"   {len(df)} filas · control {(ctl['ESTADO'] == 'OK').sum()}/{len(ctl)} OK")

    # ── 5d. PIM por función y municipalidad ──
    if "PIM_Municipalidad" in BLOQUES_ANUALES:
        print("\n📘 PIM por función y municipalidad…")
        regs = []
        with ThreadPoolExecutor(max_workers=min(WORKERS, 3)) as ex:
            for d, (r, err) in zip(deptos_nacional, ex.map(lambda d: extraer_pim_municipal(AÑO, d["codigo"]), deptos_nacional)):
                regs += r
                if err:
                    errores_totales.append({"AÑO": AÑO, "MES": "ANUAL", "NIVEL": "M", "BLOQUE": "PIM_Municipalidad", "DEPARTAMENTO": d["departamento"], "ERROR": err})
        if regs:
            df = pd.DataFrame(regs)
            archivo = os.path.join(RUTA_PIM_MUNI, "DATA", f"PIM Funcion.Municipalidad {AÑO} - {sufijo_regiones(deptos_nacional)}{SUFIJO_PRUEBA}.csv")
            df[["AÑO", "DEPARTAMENTO", "FUNCION", "MUNICIPALIDAD", "PIM"]].to_csv(archivo, index=False, encoding="utf-8-sig"); archivos.append(archivo)
            # control: suma de municipios por función vs PIM de GL por función
            ctl_filas = []
            for d in deptos_nacional:
                gl, err = extraer_pim(AÑO, "M", d["codigo"])
                gl = {x["COD_FUNCION"]: x["PIM"] for x in gl}
                suma = df[df["DEPARTAMENTO"] == d["departamento"]].groupby("COD_FUNCION")["PIM"].sum()
                for cod in sorted(set(gl) | set(suma.index)):
                    dif = float(suma.get(cod, 0.0)) - gl.get(cod, 0.0)
                    ctl_filas.append({"AÑO": AÑO, "DEPARTAMENTO": d["departamento"], "COD_FUNCION": cod, "FUNCION": f"{cod}: {FUNCIONES.get(cod, '')}",
                                      "PIM_SUMA_MUNICIPIOS": float(suma.get(cod, 0.0)), "PIM_REF_GL": gl.get(cod, 0.0), "DIF": dif,
                                      "ESTADO": "OK" if abs(dif) <= 3 else "REVISAR", "TOLERANCIA_S/": 3, "FECHA_EXTRACCION": date.today().isoformat()})
            ctl = pd.DataFrame(ctl_filas)
            archivo_ctl = os.path.join(RUTA_PIM_MUNI, "CONTROL", f"CONTROL PIM Funcion.Municipalidad {AÑO} - {sufijo_regiones(deptos_nacional)}{SUFIJO_PRUEBA}.csv")
            ctl.to_csv(archivo_ctl, index=False, encoding="utf-8-sig"); archivos.append(archivo_ctl)
            print(f"   {len(df)} filas · control {(ctl['ESTADO'] == 'OK').sum()}/{len(ctl)} OK")

    # ── 5e. Resumen ──
    print("\n" + "═" * 70 + f"\nRESUMEN {AÑO}")
    display(pd.DataFrame(resumen))
    if errores_totales:
        print("⚠️ Consultas con error:")
        with pd.option_context("display.max_colwidth", None):
            display(pd.DataFrame(errores_totales))
    if avisos_totales:
        print("⚠️ Avisos de cuadre interno (suma del detalle ≠ fila del filtro):")
        with pd.option_context("display.max_colwidth", None):
            display(pd.DataFrame(avisos_totales))
    if errores_totales or avisos_totales:
        log = pd.concat([pd.DataFrame(errores_totales), pd.DataFrame(avisos_totales)], ignore_index=True)
        log.to_csv(os.path.join(RUTA_CTL, f"LOG Scraper {AÑO} - {etiqueta_meses(meses)}{SUFIJO_PRUEBA}.csv"), index=False, encoding="utf-8-sig")
    else:
        print("✅ Sin errores y con cuadre interno en todas las consultas.")
    print(f"\n💾 {len(archivos)} archivo(s) guardados en Drive bajo {os.path.join(RUTA_FISCAL, 'Scrapeo', 'output')}")
    if PRUEBA_RAPIDA:
        print("\n🧪 Lista de verificación de la prueba (todo debe cumplirse antes de la corrida completa):")
        fila = resumen[0] if resumen else {}
        checks = [("Funcion.E > 0", fila.get("Funcion.E", 0) > 0), ("Funcion.M > 0 (detalle municipal)", fila.get("Funcion.M", 0) > 0),
                  ("Generica.E ≈ 8-10 filas", 6 <= fila.get("Generica.E", 0) <= 12), ("Fuente.E ≈ 5 filas", 3 <= fila.get("Fuente.E", 0) <= 6),
                  ("InvFuente.E > 0", fila.get("InvFuente.E", 0) > 0), ("Deuda.R > 0 (GR suele tener deuda)", fila.get("Deuda.R", 0) > 0),
                  ("Sin errores", not errores_totales), ("PIM generado", any("PIM Funcion" in a for a in archivos))]
        for nombre, ok in checks:
            print(f"   {'✅' if ok else '❌'} {nombre}")
        print("   Los archivos de prueba llevan sufijo ' - PRUEBA'; bórralos cuando ya no los necesites.")
